# 06 — Mapas de actividad acústica urbana

Análisis espacial general, sin privilegiar ninguna clase específica.

| Sección | Contenido |
|---------|-----------|
| C1 | Mapa de calor general — todos los eventos |
| C2a | Densidad por tramo 1 km — **exposición** (det/min, incluye efecto atasco) |
| C2b | Densidad por tramo 1 km — **intrínseca** (det/km, sin efecto atasco) |
| C3 | Capas por clase acústica (LayerControl, conf ≥ 0.70) |
| C4 | Comparativa ida vs vuelta ETSE (rutas mic) |
| C5 | Consistencia espacial: 6 pasadas mic |
| C6 | Trayectoria coloreada por velocidad + eventos viales |

Cada mapa se genera en dos versiones: **conf ≥ 0.10** y **conf ≥ 0.70**.

In [1]:
import sys, warnings
sys.path.insert(0, "../scripts")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import folium
from folium.plugins import HeatMap
import branca.colormap as cm
from pathlib import Path

import geo_utils as gu

# ── Umbrales de confianza ────────────────────────────────────────────────────
CONF_LOW  = 0.10
CONF_HIGH = 0.70
THRESHOLDS = [("010", CONF_LOW), ("070", CONF_HIGH)]

# ── Colores por clase acústica ───────────────────────────────────────────────
CLASS_COLORS = {
    0: "#e67e22",  # Horn
    1: "#c0392b",  # Siren
    2: "#27ae60",  # Pets
    3: "#8e44ad",  # Physiological
    4: "#2980b9",  # Speech
    5: "#f1c40f",  # Ring Tone
    6: "#1abc9c",  # Vibrating
    7: "#95a5a6",  # Notifications
    8: "#e91e63",  # Cry
}

# ── Carga de datos ───────────────────────────────────────────────────────────
geo = gu.load_geo()
geo = geo[~geo["trayecto"].isin(gu.BAD_TRAYECTOS)].copy()
geo["class_name"] = geo["class"].map(gu.CLASS_NAMES)

tracks = gu.load_tracks()
tracks = tracks[~tracks["trayecto"].isin(gu.BAD_TRAYECTOS)].copy()

center = [geo.lat.mean(), geo.lon.mean()]
Path("../outputs").mkdir(exist_ok=True)

print(f"{len(geo):,} detecciones | trayectos: {geo['trayecto'].nunique()}")
print(f"Fuentes: {sorted(geo['source'].unique())}")
geo.groupby("class_name")["confidence"].count().sort_values(ascending=False)

14,985 detecciones | trayectos: 35
Fuentes: ['mic', 'mobile']


class_name
Speech           4965
Ring Tone        3390
Vibrating        3081
Physiological    1952
Pets              958
Horn              476
Cry                93
Siren              38
Notifications      32
Name: confidence, dtype: int64

## C1 — Mapa de calor general

Kernel density sobre **todos** los eventos detectados (sin filtrar por clase).  
Peso = `confidence × min(duration_s, 5)`.

- `map_heatmap_conf010.html` — conf ≥ 0.10 (cobertura total)  
- `map_heatmap_conf070.html` — conf ≥ 0.70 (alta confianza)

In [ ]:
for tag, thr in THRESHOLDS:
    sub = geo[geo.confidence >= thr].copy()
    sub["w"] = sub["confidence"] * sub["duration_s"].clip(upper=5.0)

    m = folium.Map(location=center, zoom_start=13, tiles="cartodbpositron")
    for _, d in tracks.sort_values("time").groupby("trayecto"):
        folium.PolyLine(d[["lat", "lon"]].values.tolist(),
                        color="#999999", weight=1.5, opacity=0.35).add_to(m)
    HeatMap(sub[["lat", "lon", "w"]].values.tolist(),
            radius=16, blur=20, min_opacity=0.3, max_zoom=16).add_to(m)

    fname = f"../outputs/map_heatmap_conf{tag}.html"
    m.save(fname)
    print(f"conf≥{thr:.2f}: {len(sub):,} eventos → {fname}")
    display(m)

## C2 — Densidad por tramo 1 km

Cada trayecto se divide en segmentos de ~1 km (haversine acumulado). Para cada segmento:

| Variante | Métrica | Efecto atasco |
|----------|---------|---------------|
| **C2a — Exposición** | eventos / min (tiempo GPS) | ✅ incluido |
| **C2b — Intrínseca** | eventos / km (distancia) | ❌ eliminado |

Donde C2a > C2b → el tráfico/velocidad explica la diferencia, no la actividad acústica del tramo.

In [ ]:
# ── Funciones de segmentación ────────────────────────────────────────────────

def segment_tracks_1km(tracks_df, km=1.0, group_col="trayecto"):
    """Asigna segment_id (0,1,...) a cada trackpoint. Nuevo segmento cada ~km km."""
    result = []
    for tr, grp in tracks_df.groupby(group_col):
        grp = grp.sort_values("time").reset_index(drop=True)
        seg_id, cum_dist, seg_ids = 0, 0.0, [0]
        for i in range(1, len(grp)):
            dt_s = (grp.time.iloc[i] - grp.time.iloc[i-1]).total_seconds()
            if dt_s < 60:
                cum_dist += gu.haversine_m(
                    grp.lat.iloc[i-1], grp.lon.iloc[i-1],
                    grp.lat.iloc[i],   grp.lon.iloc[i]
                )
            if cum_dist >= km * 1000:
                seg_id += 1
                cum_dist = 0.0
            seg_ids.append(seg_id)
        grp = grp.copy()
        grp["segment_id"] = seg_ids
        grp["segment_key"] = grp.apply(
            lambda r: f"{r[group_col]}_seg{int(r['segment_id'])}", axis=1)
        result.append(grp)
    return pd.concat(result, ignore_index=True)


def gps_stats_per_segment(tracks_seg, max_gap_s=30):
    """Devuelve gps_s (tiempo GPS) y dist_km (distancia) por segment_key."""
    rows = []
    for key, grp in tracks_seg.groupby("segment_key"):
        grp = grp.sort_values("time")
        lats, lons = grp.lat.values, grp.lon.values
        t_s = d_km = 0.0
        for i in range(1, len(grp)):
            dt = (grp.time.iloc[i] - grp.time.iloc[i-1]).total_seconds()
            if 0 < dt < max_gap_s:
                t_s  += dt
                d_km += gu.haversine_m(lats[i-1], lons[i-1], lats[i], lons[i]) / 1000
        rows.append({"segment_key": key,
                     "trayecto": grp["trayecto"].iloc[0],
                     "gps_s": t_s, "dist_km": d_km})
    return pd.DataFrame(rows).set_index("segment_key")


def join_preds_to_segments(geo_df, tracks_seg, tol_s=30, group_col="trayecto"):
    """Asigna segment_key a cada predicción por cercanía temporal (merge_asof)."""
    tr_ref = tracks_seg[["time", group_col, "segment_key"]].sort_values("time").copy()
    if pd.api.types.is_datetime64tz_dtype(geo_df["t_start"]):
        if not pd.api.types.is_datetime64tz_dtype(tr_ref["time"]):
            tr_ref["time"] = tr_ref["time"].dt.tz_localize("UTC")
    out = []
    for tr, dsub in geo_df.sort_values("t_start").groupby(group_col):
        tsub = tr_ref[tr_ref[group_col] == tr]
        if tsub.empty:
            out.append(dsub.assign(segment_key=None))
            continue
        merged = pd.merge_asof(
            dsub, tsub[["time", "segment_key"]],
            left_on="t_start", right_on="time",
            direction="nearest", tolerance=pd.Timedelta(seconds=tol_s),
        ).drop(columns="time")
        out.append(merged)
    return pd.concat(out, ignore_index=True) if out else geo_df.assign(segment_key=None)


def draw_segment_map(tracks_seg, seg_stats, metric_col, caption):
    """Mapa Folium con polylines coloreadas verde→rojo por metric_col."""
    vals = seg_stats[metric_col].replace([np.inf, -np.inf], np.nan).dropna()
    if vals.empty:
        print(f"Sin datos para {metric_col}"); return None
    vmin = vals.quantile(0.05)
    vmax = vals.quantile(0.95)
    if vmax <= vmin:
        vmax = vals.max()
    cmap_fn = cm.LinearColormap(["#1a9641", "#ffffbf", "#d7191c"],
                                 vmin=vmin, vmax=vmax, caption=caption)
    m = folium.Map(location=center, zoom_start=13, tiles="cartodbpositron")
    for key, grp in tracks_seg.sort_values("time").groupby("segment_key"):
        pts = grp[["lat", "lon"]].values.tolist()
        if len(pts) < 2:
            continue
        if key in seg_stats.index and not pd.isna(seg_stats.loc[key, metric_col]):
            val = float(seg_stats.loc[key, metric_col])
            color = cmap_fn(min(max(val, vmin), vmax))
            tip = f"{key}<br>{metric_col}={val:.3f}"
        else:
            color, tip = "#cccccc", f"{key} (sin eventos)"
        folium.PolyLine(pts, color=color, weight=5, opacity=0.85,
                        tooltip=tip).add_to(m)
    cmap_fn.add_to(m)
    return m


# ── Construir segmentos ──────────────────────────────────────────────────────
tracks_seg = segment_tracks_1km(tracks)
seg_stats_base = gps_stats_per_segment(tracks_seg)
print(f"{tracks_seg['segment_key'].nunique()} segmentos | "
      f"longitud media: {seg_stats_base['dist_km'].mean():.2f} km | "
      f"tiempo medio: {seg_stats_base['gps_s'].mean():.0f} s")
seg_stats_base.describe().round(2)

In [ ]:
# ── C2a — Exposición acústica: eventos / min (tiempo GPS) ────────────────────
print("=== C2a — Exposición (eventos/min) ===\n")
for tag, thr in THRESHOLDS:
    sub = geo[geo.confidence >= thr].copy()
    geo_seg = join_preds_to_segments(sub, tracks_seg)
    evt = (geo_seg.dropna(subset=["segment_key"])
           .groupby("segment_key").size().rename("n_events"))

    seg = seg_stats_base.copy()
    seg = seg.join(evt).fillna({"n_events": 0})
    seg = seg[seg.gps_s >= 10].copy()
    seg["det_per_min"] = seg["n_events"] / (seg["gps_s"] / 60)

    m = draw_segment_map(tracks_seg, seg, "det_per_min",
                         f"Exposición acústica det/min (conf≥{thr:.2f})")
    if m:
        fname = f"../outputs/map_exposition_conf{tag}.html"
        m.save(fname)
        top = (seg[seg.n_events > 0]
               .sort_values("det_per_min", ascending=False)
               [["trayecto", "gps_s", "dist_km", "n_events", "det_per_min"]]
               .head(5).round(3))
        print(f"conf≥{thr:.2f}: {len(sub):,} eventos | "
              f"{int((seg.n_events > 0).sum())} segmentos activos")
        display(top)
        print(f"→ {fname}\n")
        display(m)

### C2b — Densidad intrínseca (eventos/km)

El denominador es la **distancia** recorrida en el segmento, fija independiente de la velocidad.  
Un atasco aumenta el tiempo pero los km permanecen iguales → el bias desaparece.

Compara con C2a: donde ambos mapas coinciden → zona intrínsecamente ruidosa.  
Donde C2a >> C2b → el tráfico lento explica la mayor exposición.

In [ ]:
# ── C2b — Densidad intrínseca: eventos / km ───────────────────────────────────
print("=== C2b — Densidad intrínseca (eventos/km) ===\n")
for tag, thr in THRESHOLDS:
    sub = geo[geo.confidence >= thr].copy()
    geo_seg = join_preds_to_segments(sub, tracks_seg)
    evt = (geo_seg.dropna(subset=["segment_key"])
           .groupby("segment_key").size().rename("n_events"))

    seg = seg_stats_base.copy()
    seg = seg.join(evt).fillna({"n_events": 0})
    seg = seg[seg.dist_km >= 0.1].copy()
    seg["det_per_km"] = seg["n_events"] / seg["dist_km"]

    m = draw_segment_map(tracks_seg, seg, "det_per_km",
                         f"Densidad intrínseca det/km (conf≥{thr:.2f})")
    if m:
        fname = f"../outputs/map_density_conf{tag}.html"
        m.save(fname)
        top = (seg[seg.n_events > 0]
               .sort_values("det_per_km", ascending=False)
               [["trayecto", "gps_s", "dist_km", "n_events", "det_per_km"]]
               .head(5).round(3))
        print(f"conf≥{thr:.2f}: {int((seg.n_events > 0).sum())} segmentos activos")
        display(top)
        print(f"→ {fname}\n")
        display(m)

## C3 — Capas por clase acústica

Mapa interactivo con LayerControl. Una capa por clase, conf ≥ 0.70.  
Tamaño del círculo proporcional a la confianza del modelo.  
Permite evaluar qué clases presentan distribución espacial coherente (y cuáles parecen FP aleatorios).

In [ ]:
sub3 = geo[geo.confidence >= CONF_HIGH].copy()

# Escala verde→rojo por confianza (igual para todas las clases)
conf_cmap = cm.LinearColormap(["#1a9641", "#ffffbf", "#d7191c"],
                               vmin=CONF_HIGH, vmax=1.0, caption="Confianza")

m3 = folium.Map(location=center, zoom_start=13, tiles="cartodbpositron")

fg_routes = folium.FeatureGroup(name="Trayectos (fondo)", show=False)
for _, d in tracks.sort_values("time").groupby("trayecto"):
    folium.PolyLine(d[["lat", "lon"]].values.tolist(),
                    color="#999", weight=1.5, opacity=0.35).add_to(fg_routes)
fg_routes.add_to(m3)

for cid, cname in gu.CLASS_NAMES.items():
    sub_c = sub3[sub3["class"] == cid]
    fg = folium.FeatureGroup(name=f"{cname} (n={len(sub_c)})",
                             show=(cid in {0, 1}))
    for _, r in sub_c.iterrows():
        col = conf_cmap(r.confidence)
        folium.CircleMarker(
            [r.lat, r.lon], radius=3 + 6 * r.confidence,
            color=col, fill=True, fill_color=col, fill_opacity=0.7,
            tooltip=f"{cname} | conf={r.confidence:.2f} | {r.trayecto}",
        ).add_to(fg)
    fg.add_to(m3)

folium.LayerControl(collapsed=False).add_to(m3)
conf_cmap.add_to(m3)
m3.save("../outputs/map_classes.html")
print("Distribución por clase (conf≥0.70):")
print(sub3.groupby("class_name").size().sort_values(ascending=False).to_string())
print("→ outputs/map_classes.html")
m3

## C4 — Comparativa ida vs vuelta: corredor ETSE

Rutas mic únicamente (6 trayectos, mismo sensor):

- **Mañana** `PAIPORTA-ETSE_1/2/3` → azul  
- **Tarde** `ETSE-PAIPORTA_1/2/3` → naranja  

Hotspots presentes en **ambas** direcciones → actividad acústica robusta, independiente del horario y sentido de la marcha.

In [7]:
etse_ids = [t for t in geo.trayecto.unique()
            if "PAIPORTA-ETSE" in t or "ETSE-PAIPORTA" in t]
print(f"Trayectos ETSE mic: {sorted(etse_ids)}")

geo_etse  = geo[geo.trayecto.isin(etse_ids) & (geo.confidence >= CONF_LOW)].copy()
trk_etse  = tracks[tracks.trayecto.isin(etse_ids)].copy()

geo_etse["direction"] = geo_etse.trayecto.apply(
    lambda t: "mañana" if t.startswith("PAIPORTA-ETSE") else "tarde")
trk_etse["direction"] = trk_etse.trayecto.apply(
    lambda t: "mañana" if t.startswith("PAIPORTA-ETSE") else "tarde")

DIR_COLORS = {"mañana": "#2980b9", "tarde": "#e67e22"}

m_etse = folium.Map(location=center, zoom_start=13, tiles="cartodbpositron")

# Trayectos coloreados por dirección
for tr, d in trk_etse.sort_values("time").groupby("trayecto"):
    direc = "mañana" if tr.startswith("PAIPORTA-ETSE") else "tarde"
    folium.PolyLine(d[["lat", "lon"]].values.tolist(),
                    color=DIR_COLORS[direc], weight=3, opacity=0.5,
                    tooltip=tr).add_to(m_etse)

# Heatmap por dirección (capas independientes)
for direc in ("mañana", "tarde"):
    sub_d = geo_etse[geo_etse.direction == direc]
    fg = folium.FeatureGroup(name=f"Eventos {direc} (n={len(sub_d)})", show=True)
    heat_pts = (sub_d.assign(w=sub_d["confidence"])
                [["lat", "lon", "w"]].values.tolist())
    HeatMap(heat_pts, radius=16, blur=20, min_opacity=0.3).add_to(fg)
    fg.add_to(m_etse)

folium.LayerControl(collapsed=False).add_to(m_etse)
m_etse.save("../outputs/map_etse_comparison.html")
print("Eventos por dirección y clase:")
print(geo_etse.groupby(["direction", "class_name"]).size()
      .unstack(fill_value=0).to_string())
print("→ outputs/map_etse_comparison.html")
m_etse

Trayectos ETSE mic: ['ETSE-PAIPORTA_1', 'ETSE-PAIPORTA_10', 'ETSE-PAIPORTA_11', 'ETSE-PAIPORTA_2', 'ETSE-PAIPORTA_3', 'ETSE-PAIPORTA_4', 'ETSE-PAIPORTA_5', 'ETSE-PAIPORTA_6', 'ETSE-PAIPORTA_7', 'ETSE-PAIPORTA_8', 'ETSE-PAIPORTA_9', 'PAIPORTA-ETSE_10', 'PAIPORTA-ETSE_11', 'PAIPORTA-ETSE_2', 'PAIPORTA-ETSE_3', 'PAIPORTA-ETSE_4', 'PAIPORTA-ETSE_5', 'PAIPORTA-ETSE_6', 'PAIPORTA-ETSE_7', 'PAIPORTA-ETSE_8', 'PAIPORTA-ETSE_9']
Eventos por dirección y clase:
class_name  Cry  Horn  Notifications  Pets  Physiological  Ring Tone  Siren  Speech  Vibrating
direction                                                                                     
mañana        3   142              8   470           1046       1440     13    1462       1673
tarde         7   111              3   306            695       1689      9     855       1394
→ outputs/map_etse_comparison.html


### C4b — Densidad intrínseca por tramo 1 km (corredor ETSE)

Mismo análisis que C2b pero restringido al corredor ETSE (21 trayectos).  
Verde = baja densidad acústica · Rojo = alta densidad.

In [ ]:
print("=== C4b — Densidad intrínseca por tramo (rutas ETSE) ===\n")
tracks_seg_etse = segment_tracks_1km(trk_etse)
seg_stats_etse  = gps_stats_per_segment(tracks_seg_etse)
print(f"{tracks_seg_etse['segment_key'].nunique()} segmentos ETSE")

for tag, thr in THRESHOLDS:
    sub_e = geo_etse[geo_etse.confidence >= thr].copy()
    geo_seg_e = join_preds_to_segments(sub_e, tracks_seg_etse)
    evt_e = (geo_seg_e.dropna(subset=["segment_key"])
             .groupby("segment_key").size().rename("n_events"))
    seg_e = seg_stats_etse.copy()
    seg_e = seg_e.join(evt_e).fillna({"n_events": 0})
    seg_e = seg_e[seg_e.dist_km >= 0.1].copy()
    seg_e["det_per_km"] = seg_e["n_events"] / seg_e["dist_km"]
    m = draw_segment_map(tracks_seg_etse, seg_e, "det_per_km",
                         f"Densidad ETSE det/km (conf≥{thr:.2f})")
    if m:
        fname = f"../outputs/map_etse_density_conf{tag}.html"
        m.save(fname)
        print(f"conf≥{thr:.2f}: {int((seg_e.n_events > 0).sum())} segmentos activos → {fname}")
        display(m)

### C4c — Capas por clase acústica (corredor ETSE, conf ≥ 0.70)

Círculos coloreados por confianza (verde→rojo). Capas separadas por clase.  
Trayectos en azul (mañana) y naranja (tarde) como referencia.

In [ ]:
sub_etse_c3 = geo_etse[geo_etse.confidence >= CONF_HIGH].copy()
conf_cmap_c4 = cm.LinearColormap(["#1a9641", "#ffffbf", "#d7191c"],
                                   vmin=CONF_HIGH, vmax=1.0, caption="Confianza")
etse_center = [trk_etse.lat.mean(), trk_etse.lon.mean()]

m_etse_c3 = folium.Map(location=etse_center, zoom_start=13, tiles="cartodbpositron")

fg_bg = folium.FeatureGroup(name="Trayectos ETSE (fondo)", show=True)
for tr, d in trk_etse.sort_values("time").groupby("trayecto"):
    direc = "mañana" if tr.startswith("PAIPORTA-ETSE") else "tarde"
    folium.PolyLine(d[["lat", "lon"]].values.tolist(),
                    color=DIR_COLORS[direc], weight=2, opacity=0.4,
                    tooltip=tr).add_to(fg_bg)
fg_bg.add_to(m_etse_c3)

for cid, cname in gu.CLASS_NAMES.items():
    sub_c = sub_etse_c3[sub_etse_c3["class"] == cid]
    fg = folium.FeatureGroup(name=f"{cname} (n={len(sub_c)})", show=(cid in {0, 1}))
    for _, r in sub_c.iterrows():
        col = conf_cmap_c4(r.confidence)
        folium.CircleMarker(
            [r.lat, r.lon], radius=3 + 6 * r.confidence,
            color=col, fill=True, fill_color=col, fill_opacity=0.7,
            tooltip=f"{cname} | conf={r.confidence:.2f} | {r.trayecto}",
        ).add_to(fg)
    fg.add_to(m_etse_c3)

folium.LayerControl(collapsed=False).add_to(m_etse_c3)
conf_cmap_c4.add_to(m_etse_c3)
m_etse_c3.save("../outputs/map_etse_classes.html")
print("→ outputs/map_etse_classes.html")
m_etse_c3

## C5 — Consistencia espacial: 6 pasadas mic

Por celda de ~100 m: ¿en cuántas de las pasadas mic (ETSE ida+vuelta) aparece al menos 1 evento?

- **Rojo oscuro** → hotspot robusto (activo en muchas pasadas)  
- **Amarillo** → evento puntual o posible artefacto

Umbral: conf ≥ 0.10 (máxima sensibilidad).

In [ ]:
CELL_SIZE = 0.001  # ~100 m (antes 0.0005 → demasiados recuadros)

mic_tray = [t for t in geo.trayecto.unique()
            if "PAIPORTA-ETSE" in t or "ETSE-PAIPORTA" in t]
n_passes = len(mic_tray)
geo_mic  = geo[geo.trayecto.isin(mic_tray) & (geo.confidence >= CONF_LOW)].copy()

geo_mic["gi"] = np.floor(geo_mic.lat / CELL_SIZE).astype(int)
geo_mic["gj"] = np.floor(geo_mic.lon / CELL_SIZE).astype(int)

cell_counts = (geo_mic.groupby(["gi", "gj", "trayecto"])
               .size().reset_index()
               .groupby(["gi", "gj"])["trayecto"].nunique()
               .rename("pass_count").reset_index())
cell_counts["lat"] = (cell_counts.gi + 0.5) * CELL_SIZE
cell_counts["lon"] = (cell_counts.gj + 0.5) * CELL_SIZE

print(f"Trayectos mic ({n_passes} pasadas)")
print(f"Celdas con ≥1 evento: {len(cell_counts)} | "
      f"Celdas en ≥{n_passes//2} pasadas: {int((cell_counts.pass_count >= n_passes//2).sum())}")
print(cell_counts.pass_count.value_counts().sort_index().to_string())

cmap5 = cm.LinearColormap(["#ffffcc", "#fd8d3c", "#800026"],
                           vmin=1, vmax=n_passes,
                           caption=f"Nº pasadas con evento (de {n_passes})")
m5 = folium.Map(location=center, zoom_start=13, tiles="cartodbpositron")
for _, r in cell_counts.iterrows():
    folium.Rectangle(
        bounds=[[r.gi * CELL_SIZE, r.gj * CELL_SIZE],
                [(r.gi + 1) * CELL_SIZE, (r.gj + 1) * CELL_SIZE]],
        fill=True, fill_color=cmap5(int(r.pass_count)),
        fill_opacity=0.75, color=None, weight=0,
        tooltip=f"{int(r.pass_count)}/{n_passes} pasadas | ({r.lat:.4f}, {r.lon:.4f})",
    ).add_to(m5)
cmap5.add_to(m5)
m5.save("../outputs/map_consistency.html")
print("→ outputs/map_consistency.html")
m5

## C6 — Trayectoria coloreada por velocidad

Velocidad en km/h coloreada (rojo = lento, verde = rápido).  
Eventos viales (Horn, Siren, conf ≥ 0.70) superpuestos como puntos.

In [ ]:
from branca.colormap import LinearColormap

tracks_sp = gu.add_speed(tracks)
tracks_sp.loc[tracks_sp.speed_kmh > 120, "speed_kmh"] = np.nan

# ── Selección de trayecto ────────────────────────────────────────────────────
# Cambia EXAMPLE_TRAYECTO al nombre deseado, o deja None para el más frecuente
EXAMPLE_TRAYECTO = None

available = sorted(tracks_sp.trayecto.unique())
print(f"Trayectos disponibles ({len(available)}):")
for i, t in enumerate(available):
    print(f"  {i+1:2d}. {t}")

example = EXAMPLE_TRAYECTO if EXAMPLE_TRAYECTO else tracks_sp.trayecto.value_counts().index[0]
print(f"\nMostrando: {example}")

seg6 = tracks_sp[tracks_sp.trayecto == example].sort_values("time")
road_events = geo[(geo["class"].isin(gu.ROAD_CLASSES)) & (geo.confidence >= CONF_HIGH)]

m6 = folium.Map(location=[seg6.lat.mean(), seg6.lon.mean()],
                zoom_start=14, tiles="cartodbpositron")
vmap = LinearColormap(["#d7191c", "#ffffbf", "#1a9641"], vmin=0, vmax=80,
                      caption="velocidad km/h")
pts = seg6[["lat", "lon"]].values
for i in range(len(pts) - 1):
    v = seg6.speed_kmh.iloc[i + 1]
    folium.PolyLine(pts[i:i+2].tolist(),
                    color=vmap(0 if pd.isna(v) else min(v, 80)),
                    weight=5, opacity=0.9).add_to(m6)
for _, r in road_events[road_events.trayecto == example].iterrows():
    folium.CircleMarker(
        [r.lat, r.lon], radius=6, color="black",
        fill=True, fill_color=CLASS_COLORS.get(r["class"], "#000"), fill_opacity=1,
        tooltip=f"{r.class_name} conf={r.confidence:.2f}",
    ).add_to(m6)
vmap.add_to(m6)
m6.save("../outputs/map_speed_trajectory.html")
print(f"→ outputs/map_speed_trajectory.html")
m6

In [ ]:
m_all_speed = folium.Map(location=center, zoom_start=12, tiles="cartodbpositron")
vmap_all = LinearColormap(["#d7191c", "#ffffbf", "#1a9641"], vmin=0, vmax=80,
                           caption="velocidad km/h")
for tr, grp in tracks_sp.groupby("trayecto"):
    grp_s = grp.sort_values("time")
    pts_all = grp_s[["lat", "lon"]].values
    for i in range(len(pts_all) - 1):
        v = grp_s.speed_kmh.iloc[i + 1]
        folium.PolyLine(
            pts_all[i:i+2].tolist(),
            color=vmap_all(0 if pd.isna(v) else min(v, 80)),
            weight=3, opacity=0.65,
            tooltip=tr,
        ).add_to(m_all_speed)
vmap_all.add_to(m_all_speed)
m_all_speed.save("../outputs/map_speed_all.html")
print("→ outputs/map_speed_all.html")
m_all_speed

### C6b — Velocidad en todos los trayectos

Todos los tracks superpuestos, coloreados por velocidad (rojo=lento, verde=rápido).  
Útil para identificar zonas de congestión sistemática en todo el dataset.

### Salidas del notebook

| Archivo | Descripción |
|---------|-------------|
| `map_heatmap_conf010.html` | Calor general, conf ≥ 0.10 |
| `map_heatmap_conf070.html` | Calor general, conf ≥ 0.70 |
| `map_exposition_conf010.html` | Exposición acústica (det/min), conf ≥ 0.10 |
| `map_exposition_conf070.html` | Exposición acústica (det/min), conf ≥ 0.70 |
| `map_density_conf010.html` | Densidad intrínseca (det/km), conf ≥ 0.10 |
| `map_density_conf070.html` | Densidad intrínseca (det/km), conf ≥ 0.70 |
| `map_classes.html` | Capas por clase (conf ≥ 0.70) |
| `map_etse_comparison.html` | Comparativa ida vs vuelta ETSE |
| `map_consistency.html` | Consistencia en 6 pasadas mic |
| `map_speed_trajectory.html` | Velocidad + eventos viales |